# Tafel / Butler–Volmer kinetics from galvanostatic pulses

Everything is set in **`tafel.yaml`**: samples and their pulse folders, the electrode (area or particle), which currents to use, the overpotential reference, IR correction.

For each state of charge, four estimates of the exchange current density i₀ are shown side by side:

| method | what it does |
|---|---|
| `excel` | the hand-built workbook method, but objective: α fixed (0.5), both lines through the OCV, intercept by least squares instead of by eye |
| `butler_volmer` | full Butler–Volmer fit to **all** points (i₀, α_a, α_c free, with errors) |
| `tafel_region` | straight lines only where the Tafel approximation holds (\|η\| ≥ 50 mV) |
| `linear` | micro-polarisation near the OCV (\|η\| ≤ 15 mV) |

**Read the `R_ir_Mohm` column.** At nA currents an ohmic drop of a few MΩ is a large part of the measured overpotential; it flattens the curves (apparent α ≪ 0.5). Compare `ir_correction: none` with `ir_correction: auto` (or an EIS resistance).

Outputs: `Exports/Tafel_<batch>.xlsx` (results, points, check against the workbook) · `Figures/Tafel/<batch>/` · each figure's data in `Exports/PlotData/Tafel/<batch>/`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from battery_analysis.project import Project

project = Project("..")
project.use("tafel", "tafel.yaml")
BATCH = "Batch1"                     # the batch this notebook works on
print("batches:", list(project._get("tafel").batches))

## 1. Analyse (writes the Excel file)

In [ ]:
samples = project.fit_tafel(batch=BATCH)
results = pd.concat([s.results() for s in samples], ignore_index=True)
results.pivot_table(index=["sample", "state"], columns="method", values="i0_mA_cm2").round(4)

## 2. Rct, transfer coefficients and fit quality

In [ ]:
cols = ["sample", "state", "OCV_V", "R_ir_Mohm", "method", "i0_mA_cm2", "Rct_ohm_cm2", "alpha_a", "alpha_c", "rms_log", "n_points", "note"]
results[[c for c in cols if c in results]].round(4)

## 3. Check against the hand-built workbook (samples with `reference_workbook:`)

In [ ]:
from battery_analysis.analysis.tafel.analysis import compare_with_reference
for s in samples:
    c = compare_with_reference(s)
    if not c.empty:
        display(c.round(4))

## 4. Figures → `Figures/Tafel/<batch>/`

In [ ]:
written = project.plot_tafel(batch=BATCH)

In [ ]:
# preview: the summary figures and one state of the first sample
from battery_analysis.analysis.tafel import plot as tp
figs = project._get("tafel").cfg.get("figures") or {}
tp.plot_summary(samples, "i0", figs); plt.show()
tp.plot_summary(samples, "Rct", figs); plt.show()
if samples and samples[0].states:
    tp.plot_state(samples[0], samples[0].states[len(samples[0].states) // 2], figs); plt.show()

## 5. Every batch at once

In [ ]:
# project.run_all(techniques=["tafel"])